In [1]:
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
load_dotenv(override=True)

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-5"

c:\Users\goran\AppData\Local\Programs\Python\Python313\Lib\ssl.py:524: UserWarning: Bad certificate in Windows certificate store: not enough data: cadata does not contain a certificate (_ssl.c:4218)
  warnings.warn(f"Bad certificate in Windows certificate store: {exc!s}")


In [3]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "extra_body": {"temperature": temperature},
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

In [ ]:
import json

def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "format": "python" | "json" | "regex",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [5]:
dataset = generate_dataset()
print(dataset)

[{'task': "Write a Python function that parses an ARN (Amazon Resource Name) and returns a dictionary with its components: partition, service, region, account-id, and resource. The function should handle ARNs in the format 'arn:partition:service:region:account-id:resource'.", 'format': 'python'}, {'task': "Create a JSON object representing an AWS IAM policy that allows read-only access to a specific S3 bucket named 'my-data-bucket'. The policy should allow s3:GetObject and s3:ListBucket actions.", 'format': 'json'}, {'task': "Write a regex pattern that validates AWS Security Group IDs. Security Group IDs start with 'sg-' followed by either 8 hexadecimal characters (old format) or 17 hexadecimal characters (new format).", 'format': 'regex'}]


In [6]:
with open('lesson11-dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)